<a href="https://colab.research.google.com/github/AliefReefandi/Practical-Statistics-for-Data-Scientists/blob/main/Chapter_8_Recommender_Systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 8 — Recommender Systems

Chapter ini membahas **Recommender Systems**, yaitu sistem yang digunakan untuk memberikan rekomendasi kepada pengguna berdasarkan informasi mengenai pengguna, item, dan interaksi yang terjadi di antara keduanya.

Recommender system banyak digunakan pada layanan seperti:

- E-commerce
- Streaming film dan musik
- Media sosial
- Marketplace
- Platform berita

Tujuan utamanya adalah membantu menemukan item yang kemungkinan relevan atau disukai oleh pengguna.

## 1. Tujuan Pembelajaran

Setelah mempelajari chapter ini, diharapkan dapat:

- Memahami konsep dasar Recommender Systems.
- Memahami hubungan antara user dan item.
- Memahami konsep similarity dalam sistem rekomendasi.
- Memahami collaborative filtering.
- Memahami bagaimana rekomendasi dapat dibuat berdasarkan user yang memiliki pola preferensi yang mirip.
- Memahami keterbatasan dan permasalahan dalam recommender systems.

## 2. Ringkasan Chapter

Recommender System merupakan sistem yang digunakan untuk memberikan rekomendasi item kepada pengguna.

Contoh sederhananya adalah ketika seorang pengguna memberikan rating terhadap beberapa film. Sistem dapat menggunakan informasi tersebut untuk menemukan pengguna lain yang memiliki pola rating yang mirip.

Berdasarkan kemiripan tersebut, sistem dapat memperkirakan item yang mungkin disukai oleh pengguna.

Secara umum, proses rekomendasi dapat digambarkan sebagai:

User → Interaksi/Rating → Similarity → Recommendation

## 3.1 User dan Item

Recommender system biasanya memiliki dua komponen utama:

- **User** → orang yang menerima rekomendasi.
- **Item** → objek yang direkomendasikan.

Contohnya pada aplikasi film:

| User | Item |
|---|---|
| User 1 | Film A |
| User 1 | Film B |
| User 2 | Film A |
| User 2 | Film C |

Data interaksi antara user dan item dapat berupa:

- Rating.
- Pembelian.
- Klik.
- Like.
- View.
- Riwayat interaksi lainnya.

Data tersebut digunakan untuk mengetahui preferensi pengguna.


## 3.2 Rating Matrix

Salah satu cara untuk merepresentasikan data recommender system adalah menggunakan **rating matrix**.

Baris menunjukkan user, sedangkan kolom menunjukkan item.

Contoh:

| | Film A | Film B | Film C | Film D |
|---|---:|---:|---:|---:|
| User 1 | 5 | 4 | - | 2 |
| User 2 | 5 | 5 | 4 | - |
| User 3 | 1 | 2 | 5 | 4 |

Tanda `-` menunjukkan bahwa user belum memberikan rating terhadap item tersebut.

Rating matrix biasanya memiliki banyak nilai yang kosong karena setiap user hanya berinteraksi dengan sebagian kecil item.

In [1]:
import pandas as pd

ratings = pd.DataFrame({
    'Film_A': [5, 5, 1],
    'Film_B': [4, 5, 2],
    'Film_C': [None, 4, 5],
    'Film_D': [2, None, 4]
}, index=['User_1', 'User_2', 'User_3'])

ratings

,Film_A,Film_B,Film_C,Film_D
User_1,5,4,NaN,2.0
User_2,5,5,4.0,NaN
User_3,1,2,5.0,4.0


## 3.3 Similarity

**Similarity** digunakan untuk mengukur seberapa mirip dua user atau dua item.

Misalnya:

User 1:

- Film A = 5
- Film B = 4
- Film D = 2

User 2:

- Film A = 5
- Film B = 5

Keduanya memiliki pola rating yang relatif mirip.

Informasi similarity tersebut dapat digunakan untuk mencari user yang memiliki preferensi serupa.

Salah satu konsep similarity yang umum digunakan pada data numerik adalah **cosine similarity**.

In [2]:
from sklearn.metrics.pairwise import cosine_similarity

# Mengisi rating yang kosong dengan 0
rating_matrix = ratings.fillna(0)

similarity = cosine_similarity(rating_matrix)

similarity_df = pd.DataFrame(
    similarity,
    index=ratings.index,
    columns=ratings.index
)

similarity_df

,User_1,User_2,User_3
User_1,1.000000,0.825723,0.461566
User_2,0.825723,1.000000,0.635210
User_3,0.461566,0.635210,1.000000


### Analisis

Nilai similarity menunjukkan tingkat kemiripan antar-user.

Nilai yang lebih tinggi menunjukkan pola rating yang lebih mirip.

Hasil similarity dapat digunakan sebagai dasar untuk mencari user yang memiliki preferensi serupa.

Namun, penggunaan nilai 0 untuk rating yang kosong pada contoh sederhana ini hanya digunakan agar perhitungan dapat dilakukan. Pada sistem rekomendasi sebenarnya, data yang hilang perlu ditangani dengan pendekatan yang sesuai.

## 3.4 Collaborative Filtering

**Collaborative Filtering** menggunakan informasi interaksi dari banyak user untuk menghasilkan rekomendasi.

Ide dasarnya adalah:

> User yang memiliki pola preferensi yang mirip kemungkinan memiliki preferensi yang mirip terhadap item lainnya.

Contohnya:

- User A menyukai Film 1 dan Film 2.
- User B juga menyukai Film 1 dan Film 2.
- User B menyukai Film 3.
- User A belum menonton Film 3.

Sistem dapat menggunakan informasi dari User B untuk memberikan rekomendasi Film 3 kepada User A.

Collaborative filtering tidak harus mengetahui isi atau karakteristik item secara langsung. Sistem menggunakan pola interaksi yang terdapat pada data.

## 3.5 User-Based Recommendation

Pada **user-based collaborative filtering**, rekomendasi dibuat berdasarkan user lain yang memiliki pola preferensi yang mirip.

Proses sederhananya:

1. Pilih user yang ingin diberikan rekomendasi.
2. Cari user lain yang memiliki similarity tinggi.
3. Lihat item yang disukai oleh user yang mirip.
4. Cari item yang belum pernah digunakan atau dinilai oleh user target.
5. Gunakan informasi tersebut sebagai rekomendasi.

Contoh:

User A mirip dengan User B.

User B menyukai:

- Film A
- Film B
- Film C

Sedangkan User A baru menyukai:

- Film A
- Film B

Maka Film C dapat menjadi kandidat rekomendasi untuk User A.

In [3]:
# Menampilkan similarity antar-user

similarity_df

,User_1,User_2,User_3
User_1,1.000000,0.825723,0.461566
User_2,0.825723,1.000000,0.635210
User_3,0.461566,0.635210,1.000000


In [4]:
# Mencari user yang paling mirip dengan User_1

similar_users = (
    similarity_df['User_1']
    .drop('User_1')
    .sort_values(ascending=False)
)

similar_users

,User_1
User_2,0.825723
User_3,0.461566


## 3.6 Item-Based Recommendation

Selain membandingkan user, sistem rekomendasi juga dapat membandingkan item.

Pada **item-based collaborative filtering**, item yang memiliki pola interaksi serupa akan dianggap memiliki hubungan yang lebih dekat.

Contohnya:

Jika banyak user yang memberikan rating tinggi terhadap:

- Film A
- Film B

maka Film A dan Film B dapat dianggap memiliki pola preferensi yang mirip.

Jika seorang user menyukai Film A, Film B dapat menjadi kandidat rekomendasi.

In [5]:
item_similarity = cosine_similarity(rating_matrix.T)

item_similarity_df = pd.DataFrame(
    item_similarity,
    index=ratings.columns,
    columns=ratings.columns
)

item_similarity_df

,Film_A,Film_B,Film_C,Film_D
Film_A,1.000000,0.981085,0.546718,0.438357
Film_B,0.981085,1.000000,0.698430,0.533333
Film_C,0.546718,0.698430,1.000000,0.698430
Film_D,0.438357,0.533333,0.698430,1.000000


## 3.7 Membuat Rekomendasi

Setelah similarity antar-item diperoleh, kita dapat mencari item yang paling mirip dengan item yang disukai user.

Sebagai contoh, jika User 1 memberikan rating tinggi terhadap Film A, sistem dapat mencari film lain yang memiliki similarity tinggi dengan Film A.

Pendekatan ini merupakan contoh sederhana dari **item-based collaborative filtering**.

In [6]:
film_a_similarity = (
    item_similarity_df['Film_A']
    .drop('Film_A')
    .sort_values(ascending=False)
)

film_a_similarity

,Film_A
Film_B,0.981085
Film_C,0.546718
Film_D,0.438357


## 3.8 Permasalahan dalam Recommender Systems

Recommender systems memiliki beberapa permasalahan.

### 1. Sparse Data

Rating matrix biasanya memiliki banyak nilai kosong karena user hanya berinteraksi dengan sebagian kecil item.

### 2. Cold Start

Sistem akan mengalami kesulitan ketika:

- terdapat user baru yang belum memiliki riwayat interaksi, atau
- terdapat item baru yang belum memiliki rating.

### 3. Scalability

Jumlah user dan item pada sistem nyata dapat sangat besar sehingga perhitungan similarity menjadi mahal.

### 4. Popularity Bias

Item yang sudah populer dapat lebih sering direkomendasikan dibandingkan item yang kurang populer.

Karena itu, recommender system perlu mempertimbangkan karakteristik data dan tujuan penggunaan sistem.

## 3.9 Kelebihan dan Kekurangan Collaborative Filtering

### Kelebihan

- Tidak membutuhkan informasi detail mengenai isi item.
- Dapat menemukan pola preferensi dari interaksi user.
- Dapat menghasilkan rekomendasi yang bersifat personal.

### Kekurangan

- Membutuhkan data interaksi yang cukup.
- Mengalami masalah sparse data.
- Memiliki masalah cold start.
- Perhitungan dapat menjadi berat ketika jumlah user dan item sangat besar.

## 3.10 Alur Sederhana Recommender System

Alur sederhana collaborative filtering dapat digambarkan sebagai berikut:

User
↓
Data Rating / Interaksi
↓
Rating Matrix
↓
Similarity
↓
User atau Item yang Mirip
↓
Kandidat Item
↓
Recommendation

Sistem menggunakan pola interaksi sebelumnya untuk menghasilkan rekomendasi kepada user.

## 3.11 Perbandingan Pendekatan

| Pendekatan | Dasar Rekomendasi |
|---|---|
| User-Based Collaborative Filtering | User lain yang memiliki preferensi mirip |
| Item-Based Collaborative Filtering | Item lain yang memiliki pola interaksi mirip |
| Content-Based Filtering | Karakteristik atau isi item |
| Hybrid | Kombinasi beberapa pendekatan |

Chapter ini berfokus pada konsep rekomendasi berbasis pola interaksi dan similarity.

# 4. Summary dan Kesimpulan

## Summary

Pada Chapter 8 dipelajari konsep **Recommender Systems**, yaitu sistem yang digunakan untuk memberikan rekomendasi berdasarkan informasi mengenai user, item, dan interaksi di antara keduanya.

Materi utama yang dibahas:

- User dan item.
- Rating matrix.
- Similarity.
- Collaborative filtering.
- User-based recommendation.
- Item-based recommendation.
- Permasalahan sparse data.
- Cold start.
- Scalability.

## Kesimpulan

Recommender system dapat menggunakan pola interaksi antar-user dan item untuk menghasilkan rekomendasi yang relevan.

Collaborative filtering memanfaatkan kemiripan preferensi antar-user atau antar-item. User-based filtering mencari user yang memiliki pola preferensi serupa, sedangkan item-based filtering mencari item yang memiliki pola interaksi yang mirip.

Dalam penerapannya, recommender system perlu memperhatikan masalah seperti data yang sparse, cold start, dan scalability.